# IS 507 專案資料前處理、探索與品質驗證

本 notebook 將兩份原始 Parquet 轉換成兩份唯一、可直接分析的 canonical processed datasets：

- `data/processed/asi_opensource_pod_hourly_processed.parquet`
- `data/processed/asi_opensource_job_execution_summary_processed.parquet`

完整流程包含：

1. 檔案完整度、schema、缺值與例外值檢查。
2. 欄位屬性、數值分布與類別分布描述。
3. 關聯性、相關係數與 PCA 探索。
4. 缺值、異常 delay、`workload_id` 與類別欄位的處理決策。
5. 產生並重新驗證兩份單一 processed Parquet。

原始檔保持不變。下游的 R 與 DuckDB 分析只讀取 processed datasets，不需要重複執行清理規則。


In [ ]:
from pathlib import Path
import shutil

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "MID" / "data").exists():
    ROOT = ROOT / "MID"

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TEMP_DIR = DATA_DIR / ".duckdb_tmp"

POD_RAW = DATA_DIR / "asi_opensource_pod_hourly_day0_29.parquet"
JOB_RAW = DATA_DIR / "asi_opensource_job_execution_summary.parquet"
POD_PROCESSED = PROCESSED_DIR / "asi_opensource_pod_hourly_processed.parquet"
JOB_PROCESSED = PROCESSED_DIR / "asi_opensource_job_execution_summary_processed.parquet"
WORKLOAD_LOOKUP = PROCESSED_DIR / "workload_id_lookup.parquet"

assert POD_RAW.exists(), f"Missing raw pod file: {POD_RAW}"
assert JOB_RAW.exists(), f"Missing raw summary file: {JOB_RAW}"

# Development profile scope. Full materialization still processes every row.
PROFILE_POD_DAY = 0
PROFILE_POD_HOUR = 0
SUMMARY_PROFILE_ROWS = 1_000_000
EDA_SAMPLE_ROWS = 50_000

# Final preprocessing decisions. Review these before enabling the write step.
WORKLOAD_ID_POLICY = "keep"       # keep, zero, drop, ordinal
CATEGORICAL_NULL_POLICY = "unknown"
WRITE_PROCESSED_FILES = False      # Change to True only after decisions are final.
OVERWRITE_PROCESSED_FILES = False

TEMP_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='8GB'")
con.execute(f"SET temp_directory='{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order=true")

con.execute(f'''CREATE OR REPLACE VIEW pod_raw AS
                SELECT * FROM read_parquet('{POD_RAW.as_posix()}')''')
con.execute(f'''CREATE OR REPLACE VIEW job_raw AS
                SELECT * FROM read_parquet('{JOB_RAW.as_posix()}')''')
con.execute(f'''CREATE OR REPLACE VIEW pod_profile_raw AS
                SELECT * FROM pod_raw
                WHERE day = {PROFILE_POD_DAY} AND hour = {PROFILE_POD_HOUR}''')
con.execute(f'''CREATE OR REPLACE VIEW job_profile_raw AS
                SELECT * FROM job_raw LIMIT {SUMMARY_PROFILE_ROWS}''')

print("Raw pod:", POD_RAW)
print("Raw summary:", JOB_RAW)
print("Processed output directory:", PROCESSED_DIR)


## 1. 資料完整度與 schema 驗證

本節使用 Parquet metadata 與完整資料中的 `day` / `hour` 欄位驗證檔案是否完整。這些檢查涵蓋全部資料，不是抽樣結果。


In [ ]:
EXPECTED_POD_ROWS = 842_390_418
EXPECTED_JOB_ROWS = 40_522_321

def parquet_metadata(path):
    parquet = pq.ParquetFile(path)
    return {
        "file": path.name,
        "size_gib": round(path.stat().st_size / 1024**3, 3),
        "rows": parquet.metadata.num_rows,
        "row_groups": parquet.metadata.num_row_groups,
        "columns": parquet.metadata.num_columns,
    }

file_inventory = pd.DataFrame([
    parquet_metadata(POD_RAW),
    parquet_metadata(JOB_RAW),
])
display(file_inventory)

pod_coverage = con.execute('''
SELECT
    count(*) AS row_count,
    min(day) AS min_day,
    max(day) AS max_day,
    count(DISTINCT day) AS distinct_days,
    min(hour) AS min_hour,
    max(hour) AS max_hour,
    count(DISTINCT (day, hour)) AS distinct_day_hours
FROM pod_raw
''').df()
display(pod_coverage)

assert file_inventory.loc[0, "rows"] == EXPECTED_POD_ROWS
assert file_inventory.loc[1, "rows"] == EXPECTED_JOB_ROWS
assert pod_coverage.loc[0, "row_count"] == EXPECTED_POD_ROWS
assert pod_coverage.loc[0, "distinct_days"] == 30
assert pod_coverage.loc[0, "distinct_day_hours"] == 720
assert (pod_coverage.loc[0, "min_hour"], pod_coverage.loc[0, "max_hour"]) == (0, 23)
print("Full-file integrity checks passed.")


### 1.1 欄位定義與分析屬性

下表將原始 schema 與分析用途整合，包含資料型態、欄位角色、measurement level 與說明。這份表格直接顯示於 notebook，不另外輸出 CSV。


In [ ]:
FIELD_INFO = {
    "pod_id": ("identifier", "nominal", "Anonymous pod identifier; may repeat across hours."),
    "workload_id": ("identifier", "nominal", "Anonymous workload identifier; may be missing."),
    "server_id": ("identifier", "nominal", "Anonymous server identifier."),
    "cluster_id": ("identifier", "nominal", "Anonymous cluster identifier."),
    "state_public": ("category", "nominal", "Public pod state category."),
    "gpu_spec_public": ("category", "nominal", "Public GPU model bucket."),
    "server_gpu_count": ("capacity", "count", "GPU count on the hosting server."),
    "server_cpu_capacity_cores": ("capacity", "count", "CPU-core capacity of the server."),
    "priority_class": ("category", "ordinal", "Public scheduling-priority class."),
    "job_type_public": ("category", "nominal", "Public job-type category."),
    "model_type_public": ("category", "nominal", "Public model-type category."),
    "is_genai_request": ("category", "binary", "Whether the request is classified as GenAI."),
    "gpu_request": ("request", "continuous", "Requested GPU-equivalent count."),
    "gpu_mem_request": ("request", "continuous", "Source-reported accelerator-memory request."),
    "cpu_request_cores": ("request", "continuous", "Requested CPU cores."),
    "used_gpu_hours": ("usage", "continuous", "GPU-hours attributed to the pod-hour."),
    "avg_gpu_sm_util": ("usage", "continuous", "Source-reported average GPU SM utilization."),
    "avg_gpu_mem_gib": ("usage", "continuous", "Average GPU-memory use in GiB."),
    "avg_cpu_request_util": ("usage", "continuous", "CPU utilization normalized by request."),
    "avg_memory_util": ("usage", "continuous", "Average memory utilization."),
    "ready_status": ("outcome", "binary", "Whether the pod became ready."),
    "schedule_status": ("outcome", "binary", "Whether the pod was scheduled."),
    "schedule_delay_sec": ("outcome", "continuous", "Scheduling delay in seconds."),
    "ready_delay_sec": ("outcome", "continuous", "Delay from scheduling to ready state."),
    "duration_hours": ("outcome", "continuous", "Observed execution duration in hours."),
    "day": ("time", "ordinal", "Relative trace day, 0 through 29."),
    "hour": ("time", "ordinal", "Hour within relative day, 0 through 23."),
}

def schema_catalog(view_name, dataset):
    schema = con.execute(f"DESCRIBE {view_name}").df()
    records = []
    for row in schema.itertuples(index=False):
        role, level, description = FIELD_INFO.get(
            row.column_name,
            ("other", "unknown", "No project-specific description assigned."),
        )
        records.append({
            "dataset": dataset,
            "column": row.column_name,
            "data_type": row.column_type,
            "role": role,
            "measurement_level": level,
            "description": description,
        })
    return pd.DataFrame(records)

field_catalog = pd.concat([
    schema_catalog("pod_raw", "pod_hourly"),
    schema_catalog("job_raw", "execution_summary"),
], ignore_index=True)
display(field_catalog)


## 2. 缺值、例外值與重複鍵檢查

完整表的列數已在上一節確認。本節使用固定範圍進行互動式品質診斷：pod Day 0 / Hour 0，以及 execution summary 的前 1,000,000 列。這些比例用於發現問題與制定規則，不應直接宣稱為 30 天母體比例。


In [ ]:
pod_quality = con.execute('''
SELECT
    count(*) AS row_count,
    count(DISTINCT pod_id) AS unique_pods,
    count(*) - count(DISTINCT pod_id) AS repeated_pod_keys,
    count(*) FILTER (WHERE workload_id IS NULL OR trim(workload_id) = '') AS missing_workload_id,
    count(*) FILTER (WHERE avg_gpu_sm_util IS NULL) AS missing_avg_gpu_sm_util,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM pod_profile_raw
''').df()

job_quality = con.execute('''
SELECT
    count(*) AS row_count,
    count(DISTINCT pod_id) AS unique_pods,
    count(*) - count(DISTINCT pod_id) AS repeated_pod_keys,
    count(*) FILTER (WHERE workload_id IS NULL OR trim(workload_id) = '') AS missing_workload_id,
    count(*) FILTER (WHERE duration_hours IS NULL) AS missing_duration,
    count(*) FILTER (WHERE duration_hours < 0) AS negative_duration,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM job_profile_raw
''').df()

def quality_long(frame, dataset):
    row = frame.iloc[0]
    total = int(row["row_count"])
    return pd.DataFrame([
        {
            "dataset": dataset,
            "metric": metric,
            "count": int(value),
            "percent_of_rows": round(100 * float(value) / total, 4),
        }
        for metric, value in row.items()
    ])

quality_report = pd.concat([
    quality_long(pod_quality, f"pod Day {PROFILE_POD_DAY} Hour {PROFILE_POD_HOUR}"),
    quality_long(job_quality, f"summary first {SUMMARY_PROFILE_ROWS:,} rows"),
], ignore_index=True)
display(quality_report)


In [ ]:
def stable_field_profile(view_name, dataset):
    profile = con.execute(f"SUMMARIZE SELECT * FROM {view_name}").df()
    profile = profile[[
        "column_name", "column_type", "min", "max", "count", "null_percentage"
    ]]
    profile.insert(0, "dataset", dataset)
    return profile

field_profile = pd.concat([
    stable_field_profile("pod_profile_raw", "pod_hourly"),
    stable_field_profile("job_profile_raw", "execution_summary"),
], ignore_index=True).merge(
    field_catalog,
    left_on=["dataset", "column_name"],
    right_on=["dataset", "column"],
    how="left",
)
field_profile = field_profile.drop(columns=["column"], errors="ignore")
display(field_profile)


### 2.1 類別與數值分布

類別表呈現各欄位的主要 levels、數量與比例；數值表提供完整的非缺值數、平均數、標準差、分位數與範圍。所有結果直接顯示在 notebook。


In [ ]:
def top_categories(view_name, dataset, columns, top_n=10):
    total = con.execute(f"SELECT count(*) FROM {view_name}").fetchone()[0]
    frames = []
    for column in columns:
        frame = con.execute(f'''
            SELECT
                '{dataset}' AS dataset,
                '{column}' AS column,
                coalesce(cast("{column}" AS VARCHAR), '<NULL>') AS value,
                count(*) AS count
            FROM {view_name}
            GROUP BY "{column}"
            ORDER BY count DESC, value
            LIMIT {int(top_n)}
        ''').df()
        frame["percent"] = (100 * frame["count"] / total).round(4)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

categorical_profile = pd.concat([
    top_categories("pod_profile_raw", "pod quick profile", [
        "state_public", "gpu_spec_public", "priority_class",
        "job_type_public", "model_type_public", "is_genai_request",
        "ready_status",
    ]),
    top_categories("job_profile_raw", "summary quick profile", [
        "gpu_spec_public", "priority_class", "job_type_public",
        "model_type_public", "is_genai_request", "ready_status",
        "schedule_status",
    ]),
], ignore_index=True)
display(categorical_profile)


In [ ]:
def numeric_profile(view_name, dataset, columns):
    expressions = ["count(*) AS total_rows"]
    for column in columns:
        expressions.extend([
            f'count("{column}") AS "{column}__non_null"',
            f'min("{column}") AS "{column}__min"',
            f'quantile_cont("{column}", 0.25) AS "{column}__q25"',
            f'quantile_cont("{column}", 0.50) AS "{column}__median"',
            f'quantile_cont("{column}", 0.75) AS "{column}__q75"',
            f'quantile_cont("{column}", 0.99) AS "{column}__p99"',
            f'max("{column}") AS "{column}__max"',
            f'avg("{column}") AS "{column}__mean"',
            f'stddev_pop("{column}") AS "{column}__sd"',
        ])
    row = con.execute(
        f"SELECT {', '.join(expressions)} FROM {view_name}"
    ).df().iloc[0]
    total = int(row["total_rows"])
    records = []
    for column in columns:
        records.append({
            "dataset": dataset,
            "column": column,
            "non_null_count": int(row[f"{column}__non_null"]),
            "non_null_percent": 100 * row[f"{column}__non_null"] / total,
            "min": row[f"{column}__min"],
            "q25": row[f"{column}__q25"],
            "median": row[f"{column}__median"],
            "q75": row[f"{column}__q75"],
            "p99": row[f"{column}__p99"],
            "max": row[f"{column}__max"],
            "mean": row[f"{column}__mean"],
            "sd": row[f"{column}__sd"],
        })
    result = pd.DataFrame(records)
    numeric_columns = result.select_dtypes(include="number").columns
    result[numeric_columns] = result[numeric_columns].round(6)
    return result

numeric_distribution = pd.concat([
    numeric_profile("pod_profile_raw", "pod quick profile", [
        "gpu_request", "gpu_mem_request", "cpu_request_cores",
        "used_gpu_hours", "avg_gpu_sm_util", "avg_gpu_mem_gib",
        "avg_cpu_request_util", "avg_memory_util",
        "schedule_delay_sec", "ready_delay_sec",
    ]),
    numeric_profile("job_profile_raw", "summary quick profile", [
        "gpu_request", "duration_hours", "schedule_delay_sec", "ready_delay_sec",
    ]),
], ignore_index=True)
display(numeric_distribution)


## 3. 探索性資料分析：關聯、群組差異與 PCA

本節只用固定的小型樣本探索變數關係，不直接改變 canonical dataset。PCA 前會以樣本中位數補數值缺值並標準化；這項補值只用於 PCA，不會自動套用到最終資料。


In [ ]:
pod_eda = con.execute(f'''
SELECT
    gpu_request, gpu_mem_request, cpu_request_cores, used_gpu_hours,
    avg_gpu_sm_util, avg_gpu_mem_gib, avg_cpu_request_util, avg_memory_util,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END AS ready_delay_sec_clean
FROM pod_profile_raw
ORDER BY hash(pod_id)
LIMIT {EDA_SAMPLE_ROWS}
''').df()

job_eda = con.execute(f'''
SELECT
    gpu_request, duration_hours,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END AS ready_delay_sec_clean
FROM job_profile_raw
ORDER BY hash(pod_id)
LIMIT {EDA_SAMPLE_ROWS}
''').df()

pod_correlation = pod_eda.corr(method="pearson", min_periods=500).round(3)
job_correlation = job_eda.corr(method="pearson", min_periods=500).round(3)

print("Pod numeric correlation matrix")
display(pod_correlation)
print("Execution-summary numeric correlation matrix")
display(job_correlation)


In [ ]:
relationship_profile = con.execute('''
SELECT
    job_type_public,
    model_type_public,
    count(*) AS rows,
    avg(CASE WHEN ready_status THEN 1.0 WHEN NOT ready_status THEN 0.0 END) AS ready_rate,
    median(duration_hours) AS median_duration_hours,
    quantile_cont(duration_hours, 0.90) AS p90_duration_hours,
    avg(gpu_request) AS mean_gpu_request
FROM job_profile_raw
GROUP BY job_type_public, model_type_public
HAVING count(*) >= 100
ORDER BY rows DESC
''').df()
display(relationship_profile)


In [ ]:
def pca_profile(frame, dataset, max_components=5):
    numeric = frame.replace([np.inf, -np.inf], np.nan).copy()
    medians = numeric.median(numeric_only=True)
    numeric = numeric.fillna(medians)
    standard_deviation = numeric.std(ddof=0)
    usable = standard_deviation[standard_deviation > 0].index.tolist()
    standardized = (
        (numeric[usable] - numeric[usable].mean())
        / numeric[usable].std(ddof=0)
    ).astype("float64")
    _, singular_values, components = np.linalg.svd(
        standardized.to_numpy(),
        full_matrices=False,
    )
    explained = singular_values**2 / np.sum(singular_values**2)
    component_count = min(max_components, len(usable))
    variance = pd.DataFrame({
        "dataset": dataset,
        "component": [f"PC{i + 1}" for i in range(component_count)],
        "explained_variance_ratio": explained[:component_count],
        "cumulative_variance": np.cumsum(explained[:component_count]),
    }).round(4)
    loadings = pd.DataFrame(
        components[:component_count].T,
        index=usable,
        columns=[f"PC{i + 1}" for i in range(component_count)],
    ).reset_index(names="feature")
    loadings.insert(0, "dataset", dataset)
    return variance, loadings.round(4), medians

pod_pca_variance, pod_pca_loadings, pod_pca_medians = pca_profile(
    pod_eda, "pod quick sample"
)
job_pca_variance, job_pca_loadings, job_pca_medians = pca_profile(
    job_eda, "summary quick sample"
)

print("PCA explained variance")
display(pd.concat([pod_pca_variance, job_pca_variance], ignore_index=True))
print("PCA loadings")
display(pd.concat([pod_pca_loadings, job_pca_loadings], ignore_index=True))


## 4. Canonical preprocessing 決策

最終 processed datasets 採用保守且可追溯的規則：保留原始欄位，另增清理欄位與缺值 indicator。一般性前處理不對 outcome、duration 或 utilization 做平均數補值；這些補值應在特定模型的 training split 中完成。

`WORKLOAD_ID_POLICY` 是產生最終檔案前最重要的決策。`drop` 只會影響 processed output，不會刪除 raw file。`ordinal` 會建立小型 lookup，已知 ID 編為正整數，缺值編為 0。


In [ ]:
processing_plan = pd.DataFrame([
    ["workload_id", "missing or blank", WORKLOAD_ID_POLICY, "Configurable ID policy; preserve missing indicator"],
    ["schedule_delay_sec", "negative sentinel / missing", "negative -> NULL in *_clean", "Negative values are not valid elapsed time"],
    ["ready_delay_sec", "negative sentinel / missing", "negative -> NULL in *_clean", "Negative values are not valid elapsed time"],
    ["duration_hours", "right-skewed", "retain; add log-ready documentation", "No invalid negative values observed"],
    ["categorical fields", "NULL and source unknown labels", CATEGORICAL_NULL_POLICY, "Create *_clean fields; preserve raw labels"],
    ["GPU utilization", "structural missingness", "retain NULL; add observed flag", "Zero-GPU requests should not receive utilization imputation"],
    ["numeric extremes", "large valid-looking values", "retain and flag", "Avoid undocumented clipping"],
    ["duplicate-looking keys", "repeated IDs", "retain", "Repeated IDs may reflect different hours or executions"],
])
processing_plan.columns = ["field", "issue", "canonical_rule", "rationale"]
display(processing_plan)


In [ ]:
def build_workload_lookup(output_path=WORKLOAD_LOOKUP):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(".part.parquet")
    temporary.unlink(missing_ok=True)
    con.execute(f'''
        COPY (
            WITH ids AS (
                SELECT nullif(trim(workload_id), '') AS workload_id_clean
                FROM pod_raw
                WHERE nullif(trim(workload_id), '') IS NOT NULL
                UNION
                SELECT nullif(trim(workload_id), '') AS workload_id_clean
                FROM job_raw
                WHERE nullif(trim(workload_id), '') IS NOT NULL
            )
            SELECT
                workload_id_clean,
                row_number() OVER (ORDER BY workload_id_clean) AS workload_id_ordinal
            FROM ids
        ) TO '{temporary.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')
    temporary.replace(output_path)
    return output_path

def workload_sql(policy):
    valid = {"keep", "zero", "drop", "ordinal"}
    if policy not in valid:
        raise ValueError(f"WORKLOAD_ID_POLICY must be one of {sorted(valid)}")
    if policy == "keep":
        return "source.workload_id_clean", "", ""
    if policy == "zero":
        return "coalesce(source.workload_id_clean, '0')", "", ""
    if policy == "drop":
        return "source.workload_id_clean", "", "WHERE source.workload_id_clean IS NOT NULL"
    if not WORKLOAD_LOOKUP.exists():
        raise FileNotFoundError(
            "Set WRITE_PROCESSED_FILES=False, run build_workload_lookup(), then rerun."
        )
    join = (
        f"LEFT JOIN read_parquet('{WORKLOAD_LOOKUP.as_posix()}') AS lookup "
        "USING (workload_id_clean)"
    )
    return "coalesce(lookup.workload_id_ordinal, 0)", join, ""

workload_expression, workload_join, workload_where = workload_sql(WORKLOAD_ID_POLICY)

con.execute(f'''
CREATE OR REPLACE VIEW pod_processed_view AS
WITH source AS (
    SELECT *, nullif(trim(workload_id), '') AS workload_id_clean
    FROM pod_raw
)
SELECT
    source.*,
    source.workload_id_clean IS NULL AS workload_id_missing,
    {workload_expression} AS workload_id_processed,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END AS ready_delay_sec_clean,
    gpu_request > 0 AS is_gpu_request,
    avg_gpu_sm_util IS NOT NULL AS gpu_utilization_observed,
    coalesce(nullif(trim(state_public), ''), 'Unknown') AS state_public_clean,
    coalesce(nullif(trim(gpu_spec_public), ''), 'Unknown') AS gpu_spec_public_clean,
    coalesce(nullif(trim(priority_class), ''), 'Unknown') AS priority_class_clean,
    coalesce(nullif(trim(job_type_public), ''), 'unknown') AS job_type_public_clean,
    coalesce(nullif(trim(model_type_public), ''), 'unknown') AS model_type_public_clean
FROM source
{workload_join}
{workload_where}
''')

con.execute(f'''
CREATE OR REPLACE VIEW job_processed_view AS
WITH source AS (
    SELECT *, nullif(trim(workload_id), '') AS workload_id_clean
    FROM job_raw
)
SELECT
    source.*,
    source.workload_id_clean IS NULL AS workload_id_missing,
    {workload_expression} AS workload_id_processed,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END AS ready_delay_sec_clean,
    CASE WHEN duration_hours >= 0 THEN duration_hours END AS duration_hours_clean,
    coalesce(nullif(trim(gpu_spec_public), ''), 'Unknown') AS gpu_spec_public_clean,
    coalesce(nullif(trim(priority_class), ''), 'Unknown') AS priority_class_clean,
    coalesce(nullif(trim(job_type_public), ''), 'unknown') AS job_type_public_clean,
    coalesce(nullif(trim(model_type_public), ''), 'unknown') AS model_type_public_clean
FROM source
{workload_join}
{workload_where}
''')

print("Canonical views created with workload policy:", WORKLOAD_ID_POLICY)
display(con.execute("DESCRIBE pod_processed_view").df())
display(con.execute("DESCRIBE job_processed_view").df())


### 4.1 寫檔前驗證

此處確認清理規則確實生效，並比較 quick scope 在處理前後的列數。若使用 `drop`，列數減少是預期結果；其他策略應保留列數。


In [ ]:
summary_scope_filter = (
    "nullif(trim(workload_id), '') IS NOT NULL"
    if WORKLOAD_ID_POLICY == "drop"
    else "TRUE"
)

prewrite_validation = con.execute(f'''
SELECT
    'pod quick scope' AS dataset,
    (SELECT count(*) FROM pod_profile_raw) AS raw_rows,
    count(*) AS processed_rows,
    count(*) FILTER (WHERE workload_id_missing) AS originally_missing_workload_id,
    count(*) FILTER (WHERE schedule_delay_sec_clean IS NULL) AS null_clean_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec_clean IS NULL) AS null_clean_ready_delay
FROM pod_processed_view
WHERE day = {PROFILE_POD_DAY} AND hour = {PROFILE_POD_HOUR}
UNION ALL
SELECT
    'summary quick scope',
    {SUMMARY_PROFILE_ROWS},
    count(*),
    count(*) FILTER (WHERE nullif(trim(workload_id), '') IS NULL),
    count(*) FILTER (WHERE schedule_delay_sec IS NULL OR schedule_delay_sec < 0),
    count(*) FILTER (WHERE ready_delay_sec IS NULL OR ready_delay_sec < 0)
FROM job_profile_raw
WHERE {summary_scope_filter}
''').df()
display(prewrite_validation)

if WORKLOAD_ID_POLICY == "drop":
    assert (prewrite_validation["processed_rows"] <= prewrite_validation["raw_rows"]).all()
else:
    assert (prewrite_validation["processed_rows"] == prewrite_validation["raw_rows"]).all()


## 5. 產生兩份 canonical processed Parquet

確認前面所有表格與決策後，將 `WRITE_PROCESSED_FILES` 改為 `True` 並重新執行本節。這是唯一會掃描並寫出全部資料的步驟。

輸出完成後，R、DuckDB 與其他分析工具只需要讀取這兩份 processed files。若修改前處理決策，重新產生並重新打包即可。


In [ ]:
def write_single_parquet(view_name, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(".part.parquet")
    if output_path.exists() and not OVERWRITE_PROCESSED_FILES:
        raise FileExistsError(
            f"{output_path} already exists. Set OVERWRITE_PROCESSED_FILES=True to replace it."
        )
    temporary.unlink(missing_ok=True)
    con.execute(f'''
        COPY (SELECT * FROM {view_name})
        TO '{temporary.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)
    ''')
    temporary.replace(output_path)
    return parquet_metadata(output_path)

if WRITE_PROCESSED_FILES:
    free_gib = shutil.disk_usage(DATA_DIR).free / 1024**3
    if free_gib < 50:
        raise RuntimeError(f"At least 50 GiB free space is required; found {free_gib:.1f} GiB")

    processed_inventory = pd.DataFrame([
        write_single_parquet("pod_processed_view", POD_PROCESSED),
        write_single_parquet("job_processed_view", JOB_PROCESSED),
    ])
    display(processed_inventory)
else:
    print("Dry run complete. Review decisions, then set WRITE_PROCESSED_FILES=True.")


## 6. 寫檔後完整度驗證與正式資料入口

Processed files 存在時，本節會重新讀取它們，確認列數、day/hour coverage、schema 與 workload policy。驗證通過後，這兩份檔案就是唯一正式的下游資料入口。


In [ ]:
if POD_PROCESSED.exists() and JOB_PROCESSED.exists():
    con.execute(f'''CREATE OR REPLACE VIEW pod_processed_file AS
                    SELECT * FROM read_parquet('{POD_PROCESSED.as_posix()}')''')
    con.execute(f'''CREATE OR REPLACE VIEW job_processed_file AS
                    SELECT * FROM read_parquet('{JOB_PROCESSED.as_posix()}')''')

    processed_file_inventory = pd.DataFrame([
        parquet_metadata(POD_PROCESSED),
        parquet_metadata(JOB_PROCESSED),
    ])
    processed_coverage = con.execute('''
        SELECT
            count(*) AS row_count,
            min(day) AS min_day,
            max(day) AS max_day,
            count(DISTINCT (day, hour)) AS distinct_day_hours
        FROM pod_processed_file
    ''').df()
    display(processed_file_inventory)
    display(processed_coverage)

    assert processed_coverage.loc[0, "min_day"] == 0
    assert processed_coverage.loc[0, "max_day"] == 29
    assert processed_coverage.loc[0, "distinct_day_hours"] == 720
    print("Processed datasets are ready for R and DuckDB.")
else:
    print("Processed files have not been materialized yet.")


## 最終資料規格

- 正式輸出永遠只有兩份 processed Parquet：pod hourly 與 execution summary。
- 原始大型 Parquet 僅作為可追溯來源，不提供下游分析直接使用。
- 欄位 profile、類別分布、numeric profile、correlation、relationship tables 與 PCA 全部直接保留在 notebook。
- 不輸出額外 profile CSV。
- R 與 DuckDB 直接讀取 `data/processed/` 內的兩份資料，不再重複執行補值、刪列或 ID 轉換。
